---
format:
  html:
    code-fold: true
jupyter: python3
---

### **Cell 1: Setup and Tokenizer Plan**
- Data Plan
  - The Tiny Shakespeare Dataset will loaded in as a .txt file. The data will be put into an array then split in order to create an iterator that can be fed into the tokenizers for training.
- Tokenizer Plan
  - Word_Tokenizer: build from scratch, taking each character in each word and creating an appended token for each word. Train it on the full dataset to create a vocab size. This tokenizer was chosen because it tokenizes full words, allowing us to use a larger vocab size.
  - BPE: Load tokenizer from the HuggingFace Hub and use the built in trainer to train tokenizer on full dataset. Use sepcial tokens to account for edge cases such as coming across a word that doesn't have an ID. BPE was chosen because it is a sub-word tokenizer, allowing the benefits of character tokenization and word tokenization without the negatives.
  - SentencePiece: Load tokenizer from HuggingFace Hub  and use the built in trainer to train tokenizer on full dataset. This will also use special tokens like BPE. SentencePiece combines the BPE and WordPiece techniques and is an interesting combination of the two.
- Training Plan
  - The dataset is split into training and testing data (90% to 10%). The data is then run through a function to create smaller, manageable chunks to run through the training and testing functions. Once the dataloaders have been defined, the training data is used only in the training function and the testing data is only used in the testing function. Both function report overall loss and accuracy.


In [5]:
# Cell 2: Data, Tokenizers, and Training Functions
import os
import requests
import random
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
import sentencepiece as spm
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler

# Check if MPS is available
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Load the Data
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text
text_iterator = text.splitlines()
random.seed(42)

# Implement Word Tokenizer
def Word_Tokenizer(text):
    tokens = []
    current_word = ""

    for char in text:
        #If character is letter or number
        if char.isalnum():
            current_word += char
        #If chracter is inside word
        elif char == "'" and current_word:
            current_word += char
        #Space or punctuation outside word
        else:
            if current_word:
                tokens.append(current_word)
                current_word = ""
            if not char.isspace():
                tokens.append(char)

    if current_word:
        tokens.append(current_word)
    return tokens

all_tokens = Word_Tokenizer(text)
# Build the vocabulary
Word_vocab = sorted(list(set(all_tokens)))
WordVocab_size = len(Word_vocab)
stoi = { token:i for i, token in enumerate(Word_vocab) }
itos = { i:token for i, token in enumerate(Word_vocab) }
print(f"Word Tokenizer Vocabulary size (unique tokens): {WordVocab_size}")

# Load BPE Tokenizer
BPE_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
BPE_tokenizer.pre_tokenizer = ByteLevel()
BPE_tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=10000,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
)
BPE_tokenizer.train_from_iterator(text_iterator, trainer=trainer)
print(f"BPE Vocabulary size (unique tokens): {BPE_tokenizer.get_vocab_size()}")

# Load Sentencepiece
model_prefix = 'sp_shakespeare_bpe'
model_file = f'{model_prefix}.model'
vocab_size = 10000
text_iterator = iter(text.splitlines())
spm.SentencePieceTrainer.train(
    sentence_iterator=text_iterator,
    model_prefix=model_prefix,
    model_type='unigram',
    vocab_size=vocab_size,
    # Add special tokens
    user_defined_symbols=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
)
SP_Tokenizer = spm.SentencePieceProcessor()
model_file = f"{model_prefix}.model"
SP_Tokenizer.load(model_file)
os.remove(model_file)
os.remove(f'{model_prefix}.vocab')
print(f"SentencePiece Vocabulary size (unique tokens): {SP_Tokenizer.vocab_size()}")

sample_text = "JULIET:\nO Romeo, Romeo! wherefore art thou Romeo?"
print(f"\nSample Text:\n '{sample_text}'")
# Word Tokenizer Sample
sample_tokens = Word_Tokenizer(sample_text)
sample_ids = [stoi[token] for token in sample_tokens]
print(f"\nWord Token IDs:\n{sample_ids}")
decoded_text = ' '.join([itos[id] for id in sample_ids])
print(f"\nDecoded Word(from IDs):\n{decoded_text}")

#BPE Tokenizer Sample
encoded = BPE_tokenizer.encode(sample_text)
print(f"\nBPE Token IDs:\n{encoded.ids}")
decoded_text = BPE_tokenizer.decode(encoded.ids)
print(f"\nBPE Decoded Text:\n{decoded_text}")

#SentencePiece Tokenizer Sample
tokens = SP_Tokenizer.encode_as_pieces(sample_text)
ids = SP_Tokenizer.encode_as_ids(sample_text)
print(f"\nSP Token IDs:\n{ids}")
decoded_text = SP_Tokenizer.decode_ids(ids)
print(f"\nSP Decoded Text:\n{decoded_text}")





#Train and Evaluate Model Functions
BLOCK_SIZE = 256   # Max sequence length
BATCH_SIZE = 64

encoded = BPE_tokenizer.encode(text)
all_ids = encoded.ids
data = torch.tensor(all_ids, dtype=torch.long)

# Split into train and test
split_idx = int(0.9 * len(data))
train_data = data[:split_idx]
test_data = data[split_idx:]

# Custom Dataset
class ShakespeareDataset(torch.utils.data.Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        chunk = self.data[idx : idx + self.block_size + 1]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

# DataLoaders
train_dataset = ShakespeareDataset(train_data, BLOCK_SIZE)
test_dataset = ShakespeareDataset(test_data, BLOCK_SIZE)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=2
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False
)


def train_model(model, data_loader, optimizer, criterion, num_epochs, device):
    model.to(device)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = GradScaler()

    #Train model
    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        custom_bar_format = "{desc}: {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}"
        progress_bar = tqdm(data_loader, desc=f"Epoch {epoch}/{num_epochs}", leave=True, bar_format=custom_bar_format)
        for batch_idx, (inputs, targets) in enumerate(train_loader, start=1):
            inputs, targets = inputs.to(device), targets.to(device)

            with autocast():
                # Forward pass
                outputs = model(inputs)
                B, T, C = outputs.shape
                outputs_flat = outputs.view(B * T, C)
                targets_flat = targets.view(B * T)
                loss = criterion(outputs_flat, targets_flat)
            #Backward pass
            optimizer.zero_grad()
            #loss.backward()
            #optimizer.step()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            _, predicted = outputs.max(dim=2)
            total += targets.numel()
            correct += predicted.eq(targets).sum().item()
            progress_bar.set_postfix(batch_loss=loss.item())

        #Print average loss and accuracy at each epoch
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = correct / total
        print(f"Epoch [{epoch}/{num_epochs}]  Loss: {epoch_loss:.4f}  Accuracy: {epoch_acc:.4f}")
        scheduler.step()
    return model


#Function that tests trained model
def test_model(model, test_loader, criterion, device):
  model.to(device)
  model.eval()
  correct = 0
  total = 0
  running_loss = 0.0

  #Test model
  with torch.no_grad():
    for inputs, targets in test_loader:
      inputs, targets = inputs.to(device), targets.to(device)
      outputs = model(inputs)

      outputs_flat = outputs.view(-1, outputs.shape[-1])
      targets_flat = targets.view(-1)
      loss = criterion(outputs_flat, targets_flat)
      running_loss += loss.item()

      _, predicted = outputs.max(dim=2)
      total += targets.numel()
      correct += predicted.eq(targets).sum().item()

    #Print model accuracy and loss on test set
    accuracy = 100.0 * correct / total
    avg_loss = running_loss / len(test_loader)
    print(f"Test accuracy of model: {accuracy:.1f}%")
    print(f"Test loss of model: {avg_loss:.4f}")

    return accuracy / 100.0, avg_loss

Word Tokenizer Vocabulary size (unique tokens): 14408
BPE Vocabulary size (unique tokens): 10000
SentencePiece Vocabulary size (unique tokens): 10000

Sample Text:
 'JULIET:
O Romeo, Romeo! wherefore art thou Romeo?'

Word Token IDs:
[1404, 8, 1818, 2187, 4, 2187, 0, 14068, 3473, 12979, 2187, 10]

Decoded Word(from IDs):
JULIET : O Romeo , Romeo ! wherefore art thou Romeo ?

BPE Token IDs:
[833, 13, 0, 30, 824, 9, 824, 5, 3061, 560, 164, 824, 15]

BPE Decoded Text:
 JULIET:O Romeo, Romeo! wherefore art thou Romeo?

SP Token IDs:
[308, 291, 123, 9, 52, 321, 8, 321, 25, 1482, 199, 42, 321, 23]

SP Decoded Text:
JULIET: O Romeo, Romeo! wherefore art thou Romeo?


In [6]:
# Cell 3: Positional Encoding (From Scratch)
def PositionalEncoding(max_seq_len, d_model):

    # PE Matrix
    pe = torch.zeros(max_seq_len, d_model)
    position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
    i = torch.arange(0, d_model, 2, dtype=torch.float)
    div_term = torch.exp(i * (-math.log(10000.0) / d_model))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    return pe

d_model = 64
max_seq_len = 256
pe_matrix = PositionalEncoding(max_seq_len, d_model)

pos, dim = 5, 10
print(f"pos={pos}, dim={dim}: {pe_matrix[pos, dim].item()}")

pos, dim = 5, 11
print(f"pos={pos}, dim={dim}: {pe_matrix[pos, dim].item()}")

pos, dim = 100, 20
print(f"pos={pos}, dim={dim}: {pe_matrix[pos, dim].item()}")

pos, dim = 100, 21
print(f"pos={pos}, dim={dim}: {pe_matrix[pos, dim].item()}")

pos=5, dim=10: 0.926757276058197
pos=5, dim=11: 0.3756607174873352
pos=100, dim=20: -0.6129372715950012
pos=100, dim=21: 0.7901315689086914


In [7]:
# Cell 4: Transformer Building Blocks (From Scratch)
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()

        # Layer normalization
        self.ln = nn.LayerNorm(d_model)

        # First Linear layer
        self.fc1 = nn.Linear(d_model, 1536)

        # Activation
        self.activation = nn.GELU()

        # Dropout
        self.dropout1 = nn.Dropout(dropout)

        # Second Linear layer
        self.fc2 = nn.Linear(1536, d_model)

        # Dropout
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        residual = x

        # Layer Normalization
        x = self.ln(x)

        # Feed-Forward Network
        x = self.fc1(x)
        x = self.activation(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.dropout2(x)

        # Residual connection
        out = residual + x

        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # Output projection
        self.W_o = nn.Linear(d_model, d_model)

        # Dropout
        self.dropout = nn.Dropout(dropout)

        # Layer normalization
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.size()

        # Store residual connection
        residual = x

        # Apply layer normalization
        x = self.layer_norm(x)

        # Linear projections
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Split into multiple heads
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k)

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Causal mask
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        mask = mask.unsqueeze(0).unsqueeze(0)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)


        # Apply causal mask
        scores = scores.masked_fill(mask == 0, float('-inf'))

        # Attention
        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        attention_output = torch.matmul(attention_weights, V)

        # Combine heads
        attention_output = attention_output.transpose(1, 2)
        attention_output = attention_output.contiguous().view(batch_size, seq_len, self.d_model)

        output = self.W_o(attention_output)
        output = self.dropout(output)
        output = residual + output

        return output

In [8]:
# Cell 5: Transformer Implementation and Training
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, context_length, dropout_rate):
        super().__init__()
        self.block_size = context_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        pe_tensor = PositionalEncoding(context_length, d_model)
        self.pos_encoding = nn.Parameter(pe_tensor, requires_grad=False)
        self.MHA = MultiHeadAttention(
            d_model=d_model,
            num_heads=n_heads,
            dropout = dropout_rate
        )
        self.FeedForward = FeedForward(
            d_model=d_model,
            dropout = dropout_rate
        )

        self.n_layers = n_layers
        self.layers = nn.ModuleList()

        for _ in range(n_layers):
            self.layers.append(MultiHeadAttention(
                d_model=d_model,
                num_heads=n_heads,
                dropout=dropout_rate
            ))
            self.layers.append(FeedForward(
                d_model=d_model,
                dropout=dropout_rate
            ))

        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        tok_emb = self.token_embedding(idx)
        seq_len = tok_emb.size(1)
        x = tok_emb + self.pos_encoding[:seq_len, :]
        for i, layer in enumerate(self.layers):
            if i % 2 == 0:
                x = layer(x)
            else:
                x = layer(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        return logits

VOCAB_SIZE = BPE_tokenizer.get_vocab_size()
D_MODEL = 384
N_LAYERS = 4
N_HEADS = 12
CONTEXT_LENGTH = 256
DROPOUT = 0.1

# Training Hyperparameters
LEARNING_RATE = 6e-4
NUM_EPOCHS = 4
BATCH_SIZE = 64

# Data & System
TRAIN_SPLIT = 0.9

# --- Initialize the Model ---
model = TransformerDecoder(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    context_length=CONTEXT_LENGTH,
    dropout_rate=DROPOUT
)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\n--- Starting Training ---")
train_model(model, train_loader, optimizer, criterion, NUM_EPOCHS, device)

print("\n--- Final Evaluation (Trained Model) ---")
test_model(model, test_loader, criterion, device)



--- Starting Training ---


/tmp/ipython-input-1267026932.py:173: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 1/4: 0/4801 [00:00<?] 

/tmp/ipython-input-1267026932.py:186: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/4]  Loss: 0.8888  Accuracy: 0.8092


...

Epoch 3/4: 0/4801 [00:00<?] 

Epoch [3/4]  Loss: 0.0915  Accuracy: 0.9770


Epoch 4/4: 0/4801 [00:00<?] 

Epoch [4/4]  Loss: 0.0692  Accuracy: 0.9831

--- Final Evaluation (Trained Model) ---
Test accuracy of model: 24.5%
Test loss of model: 12.1155


(0.2451728438172598, 12.11554755894643)

### **Cell 6: Generation and Sampling Plan**
- Prompt String: *ROMEO: O blessed, blessed night! I am afeard.*
- Temperature
  - Parameter Plan: Test T=0.2 and T=0.8.
  - Hypothesis: A lower temperature will make the next index ID prediction more focused/predictable. A higher temperature will yield a more create/exploratory prediciton for the next index ID.
- Top-K
  - Parameter Plan: Test K=5 and K=20
  - Hypothesis: A low K value will narrow the choices for the next token prediciton and will likely result in the a predictable and potentially repetitive output. A high K value will do the opposite, allowing consideration of more tokens and more diversity in token choice.
- Top-P
  - Parameter Plan: Try p=0.95 and p=0.75
  - Hypothesis: A higher p value will result in a larger nucleus of potential words, including less probable ones. The output will be more diverse, creative, and less predictable. A lower p value will create a focused nucleus with only the most probable words. The output will be more predictable.


In [16]:
# Cell 7: Generation and Sampling Implementation
def sample_temperature(logits, T):
    scaled_logits = logits / T
    probs = torch.softmax(scaled_logits, dim=-1)
    next_token_id = torch.multinomial(probs, num_samples=1)
    return next_token_id

def sample_top_k(logits: torch.Tensor, k: int) -> torch.Tensor:
    top_k_values, top_k_indices = torch.topk(logits, k)
    filtered_logits = torch.full_like(logits, float('-inf'))
    filtered_logits.scatter_(dim=0, index=top_k_indices, src=top_k_values)
    probs = torch.softmax(filtered_logits, dim=-1)
    next_token_id = torch.multinomial(probs, num_samples=1)
    return next_token_id

def sample_top_p(logits: torch.Tensor, p: float) -> torch.Tensor:
    probs = torch.softmax(logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    cumulative_probs_shifted = F.pad(cumulative_probs[:-1], (1, 0), 'constant', 0)
    nucleus = cumulative_probs_shifted < p
    sorted_probs[~nucleus] = float('-inf')
    final_probs = torch.softmax(sorted_probs, dim=-1)
    next_sorted_index = torch.multinomial(final_probs, num_samples=1)
    next_token_id = torch.gather(sorted_indices, -1, next_sorted_index)
    return next_token_id

def generate(model, tokenizer, context, sampling_method,
             num_new_tokens=100, T=1.0, k=10, p=0.9):

    context_ids_list = tokenizer.encode(context).ids
    idx = torch.tensor(context_ids_list, dtype=torch.long, device = device).unsqueeze(0)
    model.eval()

    with torch.no_grad():
        for _ in range(num_new_tokens):
            if idx.shape[1] > CONTEXT_LENGTH:
                idx_cond = idx[:, -CONTEXT_LENGTH:]
            else:
                idx_cond = idx

            logits = model(idx_cond)
            last_logits = logits[:, -1, :].squeeze(0)

            if sampling_method == 'temperature':
                next_id = sample_temperature(last_logits, T)
            elif sampling_method == 'top_k':
                next_id = sample_top_k(last_logits, k)
            elif sampling_method == 'top_p':
                next_id = sample_top_p(last_logits, p)
            else:
                raise ValueError(f"Unknown sampling method: {sampling_method}")

            next_id_reshaped = next_id.view(1, 1)
            idx = torch.cat((idx, next_id_reshaped), dim=1)

    final_ids = idx.squeeze(0).tolist()
    final_text = tokenizer.decode(final_ids)
    return final_text

model.to(device)
tokenizer = BPE_tokenizer

start_context = "ROMEO: O blessed, blessed night! I am afeard."
print(f"Context: '{start_context}'")

# Temperature Sampling
print("\n1. Temperature Sampling (T=0.8)")
generated_text_temp1 = generate(
    model, tokenizer, start_context, 'temperature',
    num_new_tokens=100, T=0.8
)
print(generated_text_temp1)

# Temperature Sampling
print("\n2. Temperature Sampling (T=0.2)")
generated_text_temp2 = generate(
    model, tokenizer, start_context, 'temperature',
    num_new_tokens=100, T=0.2
)
print(generated_text_temp2)

# Top-k Sampling
print("\n3. Top-k Sampling (k=5)")
generated_text_topk1 = generate(
    model, tokenizer, start_context, 'top_k',
    num_new_tokens=100, k=5
)
print(generated_text_topk1)

# Top-k Sampling
print("\n4. Top-k Sampling (k=20)")
generated_text_topk2 = generate(
    model, tokenizer, start_context, 'top_k',
    num_new_tokens=100, k=20
)
print(generated_text_topk2)

# Top-p Sampling
print("\n5. Top-p Sampling (p=0.95)")
generated_text_topp1 = generate(
    model, tokenizer, start_context, 'top_p',
    num_new_tokens=100, p=0.95
)
print(generated_text_topp1)

print("\n6. Top-p Sampling (p=0.75)")
generated_text_topp2 = generate(
    model, tokenizer, start_context, 'top_p',
    num_new_tokens=100, p=0.75
)
print(generated_text_topp2)


Context: 'ROMEO: O blessed, blessed night! I am afeard.'

1. Temperature Sampling (T=0.8)
 ROMEO: O blessed, blessed night! I am afeard.Being in night, all this is but a dream,Too flattering-sweet to be substantial.JULIET:Three words, dear Romeo, and good night indeed.If that thy bent of love be honourable,Thy purpose marriage, send me word to-morrow,By one that I'll procure to come to thee,Where and what time thou wilt perform the rite;And all my fortunes

2. Temperature Sampling (T=0.2)
 ROMEO: O blessed, blessed night! I am afeard.Being in night, all this is but a dream,Too flattering-sweet to be substantial.JULIET:Three words, dear Romeo, and good night indeed.If that thy bent of love be honourable,Thy purpose marriage, send me word to-morrow,By one that I'll procure to come to thee,Where and what time thou wilt perform the rite;And all my fortunes

3. Top-k Sampling (k=5)
 ROMEO: O blessed, blessed night! I am afeard.Being in night, all this is but a dream,Too flattering-sweet to 

### **Cell 8: Analysis and Discussion**
- Tokenizer Comparison
  - The word tokenizer had 12 token IDS and a vocab size of 14408. The BPE tokenizer had a vocab size of 10000 and 13 token IDs. The SentencePeice Tokenizer had a vocab size of 10000 as well and 14 token IDs. All decoded outputs were similar.
  - For the final model, I went with the BPE tokenizer. This tokenizer had the same vocab size as the SentencePiece tokenizer, it needed fewer token IDs to encode the text. While the Word tokenizer had a larger vocab size and needed fewer token IDs to encode the text than BPE, there was cocern that it would not be able to handle unknown words.
- Model Performance
  - Model was trained successfully but had a loss of 12.12 and an accuracy of 24.5%. The training loss got down to 0.069. To improve this, I would definately increase the number of epochs (which was limited by time and computing power). I would also consider increasing the size off the FeedForward hidden layer as well as increase context length so the model can "see" more past information when making a prediction.
- Sampling Analysis
  - The outputs matched the hypotheses presented for all three sampling types. With a lower Temperature, we received an output that was closer to the original text. Same with a low K value and a lower p percentage. The more creative answers, that deviated from the original text sequence came from higher T, K and p values.
  - K = 10 gave the most human-like creative answer where T = 0.8 and p = 0.95 sometimes gave gibberish is place of human-like answers.
  - T = 0.5 gave more repetitive results while p = 0.75 and K = 5 gave almost the same, safe answers (most similar to the original text).
